# Reverse Photograaphy Project

In [36]:
import pygame
import ctypes
import serial
import time
from screeninfo import get_monitors
import csv
import pandas as pd

class Projections:
    def __init__(self, display_number=1, square_size=40, interval_ms=500):

        # Configure serial interface for triggering ESP32 ReadOuts
        self.ser = serial.Serial("COM4", 9600, timeout=1)

        # Configure projector interface
        pygame.init()
        monitors = get_monitors()

        # Select monitor
        self.proj = monitors[display_number]
        print(f"Configuring Projector: {self.proj.width}x{self.proj.height}")

        # Setup display window
        self.screen = pygame.display.set_mode(
            (self.proj.width, self.proj.height), pygame.NOFRAME
        )

        # Move window to correct monitor (Windows only)
        hwnd = pygame.display.get_wm_info()["window"]
        ctypes.windll.user32.MoveWindow(
            hwnd,
            self.proj.x,
            self.proj.y,
            self.proj.width,
            self.proj.height,
            True,
        )

        # Grid parameters
        self.square_size = square_size
        self.cols = self.proj.width // square_size
        self.rows = self.proj.height // square_size
        self.patterns = {}

        # CSV logging setup
        self.logfile = open("readings.csv", "w", newline="")
        self.csvwriter = csv.writer(self.logfile)
        self.csvwriter.writerow(["pattern", "reading"])  # header


    def raster_pattern(self, pattern_number):
        """Generate one pattern with a single white square at index = pattern_number"""
        surface = pygame.Surface((self.proj.width, self.proj.height))
        surface.fill((0, 0, 0))  # start black

        # Compute which cell to turn white
        x_idx = pattern_number % self.cols
        y_idx = pattern_number // self.cols
        rect = (
            x_idx * self.square_size,
            y_idx * self.square_size,
            self.square_size,
            self.square_size,
        )
        pygame.draw.rect(surface, (255, 255, 255), rect)

        return surface

    def generate_rasters(self):
        """Pre-generate all raster patterns into a dictionary"""
        total = self.cols * self.rows
        print(f"Generating {total} raster patterns...")
        self.patterns = {i: self.raster_pattern(i) for i in range(total)}

    # Runs with blocking
    def project_and_read(self, pattern_idx):
        msg = f"Projecting: {pattern_idx}\n"
        self.ser.write(msg.encode())

        # Wait for Arduino response
        line = self.ser.readline().decode().strip()
        return line

    def run_patterns(self):
        """Cycle through all patterns, blocking until ESP32 responds"""
        if not self.patterns:
            self.generate_rasters()

        current = 0
        running = True

        while running and current < len(self.patterns):
            for event in pygame.event.get():
                if event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
                    running = False

            # Draw current pattern
            self.screen.blit(self.patterns[current], (0, 0))
            pygame.display.flip()

            # Send + wait for response
            response = self.project_and_read(current)
            time.sleep(0.1)
            print(f"PC sent pattern {current}, ESP32 responded: {response}")

            self.csvwriter.writerow([current, response])
            self.logfile.flush() # likely not necessary but if i end up running long experiments, it could be useful

            # Advance AFTER ESP32 has read
            current += 1

        pygame.quit()
        self.ser.close()

        self.logfile.close()

In [37]:
Projections(display_number=1, square_size=80).run_patterns()

Configuring Projector: 1280x720
Generating 144 raster patterns...
PC sent pattern 0, ESP32 responded: Reading: 0, 4095
PC sent pattern 1, ESP32 responded: Reading: 1, 4095
PC sent pattern 2, ESP32 responded: Reading: 2, 4095
PC sent pattern 3, ESP32 responded: Reading: 3, 4095
PC sent pattern 4, ESP32 responded: Reading: 4, 4095
PC sent pattern 5, ESP32 responded: Reading: 5, 4095
PC sent pattern 6, ESP32 responded: Reading: 6, 4095
PC sent pattern 7, ESP32 responded: Reading: 7, 4095
PC sent pattern 8, ESP32 responded: Reading: 8, 4095
PC sent pattern 9, ESP32 responded: Reading: 9, 4095
PC sent pattern 10, ESP32 responded: Reading: 10, 4095
PC sent pattern 11, ESP32 responded: Reading: 11, 4095
PC sent pattern 12, ESP32 responded: Reading: 12, 4095
PC sent pattern 13, ESP32 responded: Reading: 13, 4095
PC sent pattern 14, ESP32 responded: Reading: 14, 4095
PC sent pattern 15, ESP32 responded: Reading: 15, 4095
PC sent pattern 16, ESP32 responded: Reading: 16, 4095
PC sent pattern 17,

In [25]:
import serial.tools.list_ports

ports = serial.tools.list_ports.comports()
for port, desc, hwid in sorted(ports):
    print(f"Port: {port}")
    print(f"Description: {desc}")
    print(f"Hardware ID: {hwid}\n")

Port: COM4
Description: USB Serial Device (COM4)
Hardware ID: USB VID:PID=2341:0070 SER=744DBDA074A8 LOCATION=1-1:x.1



In [8]:
import serial, time

ser = serial.Serial("COM4", 9600, timeout=1)

def project_and_read(pattern_idx):
    msg = f"Projecting: {pattern_idx}\n"
    ser.write(msg.encode())

    # Wait for Arduino response
    line = ser.readline().decode().strip()
    return line

for i in range(10):
    # (insert code to actually project pattern i here)
    response = project_and_read(i)
    print(f"PC sent pattern {i}, Arduino responded: {response}")
    time.sleep(5)  # control pacing


SerialException: could not open port 'COM4': PermissionError(13, 'Access is denied.', None, 5)